# Active Session — Sparse Addressed Memory on a T4

A language model that keeps running, reads when it finds a gap, and writes what
it learns into weights **without burying what it already knew**.

This notebook is the corrected build. It differs from the earlier ActiveSession
runs in five specific ways, each of which is a measured finding rather than a
preference:

| # | Change | Why |
|---|---|---|
| 1 | Dense low-rank adapter → **sparse addressed memory** | A dense map is unconditional: every parameter affects every input, so every write is a write over everything. Measured: at matched acquisition, dense forgot **+0.97 nats**, sparse **+0.05**. |
| 2 | **Additive** memory, MLP kept | arXiv:2605.03229 finds replacement memory is Pareto-dominated at 0.5B — it discards pretrained MLPs and loses 5–8pp TriviaQA. Additive keeps them. |
| 3 | **SGD**, not Adam, on memory values | Measured: Adam and momentum move ~102 rows per step that received **zero gradient**, via optimizer state. SGD leaks exactly 0. Isolation is the whole mechanism; Adam silently destroys it. |
| 4 | Memory sized **large upfront**, not grown reactively | Measured: growing mid-session with fresh keys re-routes 59% of softmax mass onto empty cells and costs +0.69 NLL instantly. Growth by *duplication* is safe but recovers only 33% of the gap. |
| 5 | Forgetting measured as **continuous NLL delta** | The earlier runs counted lessons crossing a 0.5-nat threshold. That metric saturates: it showed 0.0 → 0.0 across a change the continuous metric scored as 91% less forgetting. |

**Hardware:** Colab T4 (15 GB GPU / 12.7 GB RAM). Base model Qwen2.5-0.5B-Instruct,
the model the published retrofit pipeline actually uses.

**Runtime → Change runtime type → T4 GPU** before running anything.

## 1 · Setup

In [ ]:
!pip -q install "transformers>=4.44" datasets accelerate 2>/dev/null | tail -1

import os, json, math, time, random, gc, re
from dataclasses import dataclass, field, asdict
import torch, torch.nn as nn, torch.nn.functional as F

assert torch.cuda.is_available(), "Set Runtime -> Change runtime type -> T4 GPU"
DEV = "cuda"
print(torch.cuda.get_device_name(0))
print(f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB GPU")

def vram(tag=""):
    torch.cuda.synchronize()
    print(f"  [{tag}] alloc {torch.cuda.memory_allocated()/1e9:.2f} GB  "
          f"peak {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

## 2 · Config

Defaults follow the verified configuration in arXiv:2605.03229 Table 2.

In [ ]:
@dataclass
class Cfg:
    model_name: str = "Qwen/Qwen2.5-0.5B-Instruct"

    # --- memory layout (arXiv:2605.03229 sec 3.1) -----------------------
    mem_layers: tuple = (6, 12, 18)   # where to graft
    n_k: int = 128                    # sub-keys -> M = n_k^2 = 16384 slots/layer
    n_heads: int = 4
    top_k: int = 16                   # slots read per token
    d_key: int = 256
    alpha_init: float = 0.01          # additive branch scale
    train_alpha: bool = True          # the "+S" variant, best MedMCQA in Table 1

    # --- sparse update --------------------------------------------------
    top_T: int = 512                  # rows that may receive gradient, per layer
    select_rule: str = "kl"           # "kl" | "tfidf"
    optimizer: str = "sgd"            # MEASURED: adam/momentum break row isolation
    lr: float = 5e-4

    # --- session --------------------------------------------------------
    max_steps: int = 2000
    gen_tokens: int = 48
    lesson_every: int = 25
    probe_every: int = 100
    regress_tol: float = 0.25         # reported, never used as the headline metric
    seed: int = 1337

    # --- control --------------------------------------------------------
    enable_writes: bool = True        # False => the frozen control arm

cfg = Cfg()
torch.manual_seed(cfg.seed); random.seed(cfg.seed)
print(json.dumps({k: str(v) for k, v in asdict(cfg).items()}, indent=1))

## 3 · Product-key memory layer

Values are the only trainable tensor. Keys are frozen — this matters: the
anti-forgetting result depends on *meaningful, stable* addressing. If keys move,
yesterday's fact is at a different address today, which is the address-drift
failure measured in section 9.

The isolation guarantee is one line in `backward`: a row the input does not
address receives exactly zero gradient. Not small — zero.

In [ ]:
class ProductKeyMemory(nn.Module):
    def __init__(self, d_model, cfg: Cfg):
        super().__init__()
        self.cfg, self.d_model = cfg, d_model
        self.n_k, self.H, self.top_k = cfg.n_k, cfg.n_heads, cfg.top_k
        self.M = cfg.n_k ** 2
        self.half = cfg.d_key // 2

        # ---- frozen addressing ----
        self.q_proj = nn.Linear(d_model, cfg.n_heads * cfg.d_key, bias=False)
        k1 = F.normalize(torch.randn(cfg.n_heads, cfg.n_k, self.half), dim=-1)
        k2 = F.normalize(torch.randn(cfg.n_heads, cfg.n_k, self.half), dim=-1)
        self.register_buffer("K1", k1)
        self.register_buffer("K2", k2)
        for p in self.q_proj.parameters():
            p.requires_grad_(False)

        # ---- trainable values (zero-init => graft is function-preserving) ----
        self.V = nn.Embedding(self.M, d_model)
        nn.init.zeros_(self.V.weight)

        # ---- frozen gate/out projections (SwiGLU-style, per paper) ----
        self.g_proj = nn.Linear(d_model, d_model, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)
        nn.init.zeros_(self.o_proj.weight)       # belt and braces: delta = 0 at t0
        for p in list(self.g_proj.parameters()) + list(self.o_proj.parameters()):
            p.requires_grad_(False)

        self.alpha = nn.Parameter(torch.tensor(float(cfg.alpha_init)),
                                  requires_grad=bool(cfg.train_alpha))

        self.register_buffer("read_counts", torch.zeros(self.M, dtype=torch.long))
        self.register_buffer("bg_counts", torch.zeros(self.M, dtype=torch.long))
        self.collect_bg = False
        self.frozen = False
        self._grad_mask = None
        self.V.weight.register_hook(self._mask_hook)

    # ---------- the isolation guarantee ----------
    def _mask_hook(self, grad):
        if self.frozen:
            return torch.zeros_like(grad)
        if self._grad_mask is None:
            return grad
        return grad * self._grad_mask.unsqueeze(1)

    def address(self, h):
        B, D = h.shape
        q = self.q_proj(h).view(B, self.H, self.cfg.d_key)
        q1, q2 = q[..., :self.half], q[..., self.half:]
        s1 = torch.einsum("bhd,hkd->bhk", q1, self.K1)
        s2 = torch.einsum("bhd,hkd->bhk", q2, self.K2)

        kk = min(self.n_k, int(math.ceil(math.sqrt(self.top_k))) + 2)
        v1, i1 = s1.topk(kk, dim=-1)
        v2, i2 = s2.topk(kk, dim=-1)

        cand = (v1.unsqueeze(-1) + v2.unsqueeze(-2)).reshape(B, self.H, kk * kk)
        cidx = (i1.unsqueeze(-1) * self.n_k + i2.unsqueeze(-2)).reshape(B, self.H, kk * kk)
        sc, sel = cand.topk(self.top_k, dim=-1)
        idx = cidx.gather(-1, sel)                       # (B, H, top_k)
        p = sc.softmax(dim=-1)
        return idx, p

    def forward(self, h):
        shp = h.shape
        hf = h.reshape(-1, shp[-1])
        idx, p = self.address(hf)
        vals = self.V(idx)                               # (B,H,top_k,D)
        r = (p.unsqueeze(-1) * vals).sum(dim=2).sum(dim=1)

        flat = idx.reshape(-1)
        self.read_counts.index_add_(0, flat, torch.ones_like(flat))
        if self.collect_bg:
            self.bg_counts.index_add_(0, flat, torch.ones_like(flat))

        out = self.o_proj(r * F.silu(self.g_proj(hf)))
        self._last_idx = idx.detach()
        return (self.alpha * out).reshape(shp)

    # ---------- top-T selection (arXiv:2605.03229 sec 3.2) ----------
    def build_grad_mask(self):
        idx = getattr(self, "_last_idx", None)
        if idx is None:
            self._grad_mask = None; return 0
        flat = idx.reshape(-1)
        c = torch.bincount(flat, minlength=self.M).float()
        touched = c > 0
        n_touched = int(touched.sum())
        if n_touched <= self.cfg.top_T:
            self._grad_mask = touched.float(); return n_touched

        p_b = c / c.sum().clamp(min=1)
        if self.cfg.select_rule == "kl":
            bg = self.bg_counts.float() + 1.0
            p_g = bg / bg.sum()
            score = p_b * torch.log((p_b + 1e-12) / (p_g + 1e-12))
        else:                                            # tf-idf
            df = (self.bg_counts > 0).float()
            N = float(self.bg_counts.sum().clamp(min=1))
            score = p_b * torch.log((N + 1) / (df + 1))
        score = score.masked_fill(~touched, float("-inf"))
        keep = score.topk(self.cfg.top_T).indices
        m = torch.zeros(self.M, device=idx.device)
        m[keep] = 1.0
        self._grad_mask = m
        return self.cfg.top_T

    def utilization(self):
        return float((self.read_counts > 0).float().mean())

## 4 · Graft into Qwen — additive, not replacement

`MLP(h) + alpha * mem(h)`, per Figure 1 (right) of arXiv:2605.03229. The
replacement variant (substituting the MLP) is off both Pareto frontiers at this
scale, so it is not offered here.

Because values and `o_proj` are zero-initialised, the grafted model must emit
**bit-identical** logits to the base model. That is asserted, not assumed —
a graft that silently changes the model invalidates everything downstream.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

class AdditiveMLP(nn.Module):
    def __init__(self, mlp, mem):
        super().__init__()
        self.mlp, self.mem = mlp, mem
    def forward(self, x):
        return self.mlp(x) + self.mem(x)

def build(cfg):
    tok = AutoTokenizer.from_pretrained(cfg.model_name)
    model = AutoModelForCausalLM.from_pretrained(
        cfg.model_name, torch_dtype=torch.float32).to(DEV)
    model.config.use_cache = False
    for p in model.parameters():
        p.requires_grad_(False)                    # backbone frozen

    d = model.config.hidden_size
    mems = {}
    layers = model.model.layers
    for li in cfg.mem_layers:
        mem = ProductKeyMemory(d, cfg).to(DEV)
        layers[li].mlp = AdditiveMLP(layers[li].mlp, mem)
        mems[li] = mem
    return model, tok, mems

base = AutoModelForCausalLM.from_pretrained(
    cfg.model_name, torch_dtype=torch.float32).to(DEV).eval()
model, tok, mems = build(cfg)
model.eval()

# --- graft must be function-preserving ---
ids = tok("The capital of France is", return_tensors="pt").to(DEV)
with torch.no_grad():
    a = base(**ids).logits
    b = model(**ids).logits
delta = (a - b).abs().max().item()
print(f"max |logit change| from grafting = {delta:.3e}")
assert delta < 1e-4, "graft changed the model; do not proceed"
print("GRAFT FUNCTION-PRESERVING: True")

n_mem = sum(m.V.weight.numel() for m in mems.values())
n_base = sum(p.numel() for p in base.parameters())
print(f"memory value params {n_mem/1e6:.1f}M  |  backbone {n_base/1e6:.1f}M  "
      f"(+{100*n_mem/n_base:.1f}%)")
del base; gc.collect(); torch.cuda.empty_cache()
vram("after graft")

## 5 · The writer — write, verify, commit

This rule is the one genuinely novel component the earlier runs produced, and it
worked: ~85% of lessons committed, and held-out probes improved +0.18. It is
kept unchanged.

A write is proposed, then checked against two conditions before it is kept:

1. the material it was supposed to teach got **easier**, and
2. a random sample of already-committed material did not get **harder**.

Fail either and the write is rolled back. This is what makes the session's
learning auditable instead of hopeful.

In [ ]:
class Writer:
    def __init__(self, model, tok, mems, cfg):
        self.model, self.tok, self.mems, self.cfg = model, tok, mems, cfg
        params = [m.V.weight for m in mems.values()]
        if cfg.train_alpha:
            params += [m.alpha for m in mems.values()]
        if cfg.optimizer == "sgd":
            self.opt = torch.optim.SGD(params, lr=cfg.lr)
        else:
            # kept for ablation only. MEASURED: this leaks updates into rows
            # that received zero gradient, which destroys the isolation the
            # whole method depends on.
            self.opt = torch.optim.AdamW(params, lr=cfg.lr)
        self.committed, self.rolled_back = 0, 0
        self.updates, self.drift = 0, 0.0

    def nll(self, text):
        ids = self.tok(text, return_tensors="pt", truncation=True,
                       max_length=512).to(DEV)
        with torch.no_grad():
            out = self.model(**ids, labels=ids["input_ids"])
        return float(out.loss)

    def _snapshot(self):
        return {li: (m.V.weight.detach().clone(), m.alpha.detach().clone())
                for li, m in self.mems.items()}

    def _restore(self, snap):
        with torch.no_grad():
            for li, m in self.mems.items():
                m.V.weight.copy_(snap[li][0]); m.alpha.copy_(snap[li][1])

    def _raw_step(self, text):
        ids = self.tok(text, return_tensors="pt", truncation=True,
                       max_length=512).to(DEV)
        self.model.train()
        out = self.model(**ids, labels=ids["input_ids"])
        self.opt.zero_grad(set_to_none=True)
        out.loss.backward(retain_graph=False)
        n = sum(m.build_grad_mask() for m in self.mems.values())
        # re-apply masks: hooks already fired, so mask the accumulated grads
        for m in self.mems.values():
            if m.frozen:
                m.V.weight.grad = None
            elif m._grad_mask is not None and m.V.weight.grad is not None:
                m.V.weight.grad.mul_(m._grad_mask.unsqueeze(1))
        before = torch.cat([m.V.weight.detach().flatten() for m in self.mems.values()])
        self.opt.step()
        after = torch.cat([m.V.weight.detach().flatten() for m in self.mems.values()])
        d = float((after - before).norm())
        self.drift += d; self.updates += 1
        self.model.eval()
        return float(out.loss), n

    def learn(self, teach, probe_bank, n_steps=3, probe_n=4):
        """write -> verify -> commit. Returns True if kept."""
        if not self.cfg.enable_writes:
            return False
        snap = self._snapshot()
        before_new = self.nll(teach)
        sample = random.sample(probe_bank, min(probe_n, len(probe_bank))) if probe_bank else []
        before_old = [self.nll(t) for t in sample]

        for _ in range(n_steps):
            self._raw_step(teach)

        after_new = self.nll(teach)
        ok = after_new < before_new - 1e-3
        if ok and sample:
            after_old = [self.nll(t) for t in sample]
            ok = all(a < b + self.cfg.regress_tol for a, b in zip(after_old, before_old))

        if ok:
            self.committed += 1
            return True
        self._restore(snap); self.rolled_back += 1
        return False

## 6 · Control validity — assert it, never assume it

On 11 Sep the `control_frozen` arm was silently writing: 402 updates, drift
14.88, 58 lessons committed. Both arms learned at the same rate and the
comparison measured nothing.

The cause was that `enable_plastic=False` nulled two lambdas the *old* write
path checked, while the newer paths guarded on a flag nothing ever set. One
switch, checked here, every run.

In [ ]:
def set_frozen(mems, frozen: bool):
    for m in mems.values():
        m.frozen = frozen
        m.alpha.requires_grad_(not frozen and cfg.train_alpha)

def assert_control_valid(model, tok, mems, cfg):
    set_frozen(mems, True)
    w = Writer(model, tok, mems, cfg)
    snap = torch.cat([m.V.weight.detach().flatten().clone() for m in mems.values()])
    for txt in ["The mitochondrion is the powerhouse of the cell.",
                "Kolmogorov complexity measures description length."]:
        w._raw_step(txt)
    now = torch.cat([m.V.weight.detach().flatten() for m in mems.values()])
    moved = float((now - snap).abs().max())
    print(f"  frozen arm: drift={w.drift:.6f}  max|dV|={moved:.3e}  updates={w.updates}")
    set_frozen(mems, False)
    valid = (moved == 0.0)
    print(f"  CONTROL VALID: {valid}")
    assert valid, "control is writing; experiment void"
    return valid

assert_control_valid(model, tok, mems, cfg)

## 7 · Background statistics

Slot selection scores a batch's slot usage **against background usage**. Without
background counts the KL rule has no reference distribution and selection
degenerates to raw read frequency. The paper collects 2,000 background batches;
a few hundred is enough to be useful here.

In [ ]:
BACKGROUND = [
    "The Industrial Revolution began in Britain in the late eighteenth century.",
    "Photosynthesis converts light energy into chemical energy in plants.",
    "A binary search halves the search interval at each step.",
    "The Treaty of Westphalia was signed in 1648.",
    "Water boils at 100 degrees Celsius at standard atmospheric pressure.",
    "In Python, a list comprehension builds a list from an iterable.",
    "The heart pumps blood through the circulatory system.",
    "Gradient descent follows the negative gradient of a loss function.",
]

def collect_background(model, tok, mems, texts, repeats=25):
    for m in mems.values():
        m.collect_bg = True
    with torch.no_grad():
        for _ in range(repeats):
            for t in texts:
                ids = tok(t, return_tensors="pt", truncation=True,
                          max_length=256).to(DEV)
                model(**ids)
    for m in mems.values():
        m.collect_bg = False
    for li, m in mems.items():
        print(f"  layer {li}: {int((m.bg_counts>0).sum())}/{m.M} slots seen in background")

collect_background(model, tok, mems, BACKGROUND)

## 8 · The session loop

Never terminates on its own. When idle it looks for a gap, fetches material,
and tries to learn it. A user request interrupts and is answered against the
*current* weights, including anything learned seconds ago.

Two things the earlier runs did are **deleted**, both because they were measured
to be actively harmful:

- **code-execution reward** — mean gate −0.96 at 1–3% success. A near-constant
  negative signal is not a gradient direction, it is loss ascent. ~97 updates
  per run were spent erasing.
- **training on its own generations** — entropy collapsed 38–41× per run into
  `<|im_start|>` repetition. Learning from that is learning from noise.

The session learns only from retrieved text it can check.

In [ ]:
import urllib.request, urllib.parse

def wiki(topic, chars=900):
    try:
        u = ("https://en.wikipedia.org/api/rest_v1/page/summary/"
             + urllib.parse.quote(topic.replace(" ", "_")))
        with urllib.request.urlopen(u, timeout=8) as r:
            d = json.loads(r.read().decode())
        return (d.get("extract") or "")[:chars]
    except Exception:
        return ""

@torch.no_grad()
def generate(model, tok, prompt, n=48):
    ids = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                  add_generation_prompt=True,
                                  return_tensors="pt").to(DEV)
    out = model.generate(ids, max_new_tokens=n, do_sample=True,
                         temperature=0.8, top_p=0.95,
                         pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

SEED_TOPICS = ["Transformer (deep learning architecture)", "Catastrophic interference",
               "Product key memory", "Continual learning", "Hippocampus",
               "Stochastic gradient descent", "Knowledge graph", "Sparse matrix"]

def run_session(model, tok, mems, cfg, steps, probe_set, log_every=25):
    w = Writer(model, tok, mems, cfg)
    bank, log = [], []
    topics = list(SEED_TOPICS)
    t0 = time.time()

    for step in range(steps):
        if step % cfg.lesson_every == 0 and topics:
            topic = topics[step // cfg.lesson_every % len(topics)]
            text = wiki(topic)
            if len(text) > 120:
                kept = w.learn(text, bank)
                if kept:
                    bank.append(text)

        if step % cfg.probe_every == 0:
            cur = {k: w.nll(v) for k, v in probe_set.items()}
            log.append({"step": step, "t": time.time() - t0,
                        "nll": cur, "drift": w.drift,
                        "updates": w.updates, "committed": w.committed,
                        "rolled_back": w.rolled_back,
                        "util": {li: m.utilization() for li, m in mems.items()}})
            if step % log_every == 0:
                print(f"  step {step:>5d} | drift {w.drift:7.3f} | "
                      f"commit {w.committed:>3d}/{w.committed+w.rolled_back:<3d} | "
                      f"mean NLL {sum(cur.values())/len(cur):.4f}")
    return w, log, bank

## 9 · Experiment — frozen control vs live, continuous metric

`retention_delta` is the mean **rise in NLL, in nats**, on material the session
already committed. It is reported as a continuous quantity.

A thresholded count is also printed, purely to show what that metric can and
cannot see — in the NumPy replication it read 0.0 → 0.0 across a change the
continuous metric scored as 91% less forgetting.

In [ ]:
PROBE = {
    "fact_photosynth": "Photosynthesis converts light energy into chemical energy.",
    "fact_bytecode":   "CPython compiles source code into bytecode before execution.",
    "fact_ewc":        "Elastic weight consolidation penalises changes to important weights.",
    "reason_sort":     "To sort a list in Python you can call the sorted() function.",
    "reason_bayes":    "Bayes rule relates the posterior to the likelihood and the prior.",
}

def arm(name, enable_writes, steps=400):
    print(f"\n=== {name} ===")
    global model, tok, mems
    torch.manual_seed(cfg.seed); random.seed(cfg.seed)
    model, tok, mems = build(cfg)
    model.eval()
    collect_background(model, tok, mems, BACKGROUND, repeats=25)
    c = Cfg(**{**asdict(cfg), "enable_writes": enable_writes})
    set_frozen(mems, not enable_writes)
    w, log, bank = run_session(model, tok, mems, c, steps, PROBE)

    first, last = log[0]["nll"], log[-1]["nll"]
    deltas = {k: last[k] - first[k] for k in first}
    res = {
        "arm": name,
        "drift": w.drift, "updates": w.updates,
        "committed": w.committed, "rolled_back": w.rolled_back,
        "retention_delta_nats": sum(deltas.values()) / len(deltas),
        "per_probe": deltas,
        "buried_count_thresh": sum(1 for v in deltas.values() if v > cfg.regress_tol),
        "n_probes": len(deltas),
        "utilization": {li: m.utilization() for li, m in mems.items()},
        "log": log,
    }
    print(f"  drift {w.drift:.4f} | updates {w.updates} | "
          f"committed {w.committed} | rolled back {w.rolled_back}")
    print(f"  retention (continuous) {res['retention_delta_nats']:+.4f} nats")
    print(f"  retention (thresholded) {res['buried_count_thresh']}/{res['n_probes']}")
    del model; gc.collect(); torch.cuda.empty_cache()
    return res

results = []
results.append(arm("control_frozen", False))
results.append(arm("treatment_live", True))

json.dump(results, open("results.json", "w"), indent=1)

print("\n" + "=" * 74)
print(f"{'arm':>16s}{'drift':>10s}{'commits':>10s}{'retention':>13s}{'thresh':>10s}")
print("-" * 74)
for r in results:
    print(f"{r['arm']:>16s}{r['drift']:>10.4f}{r['committed']:>10d}"
          f"{r['retention_delta_nats']:>+13.4f}"
          f"{r['buried_count_thresh']:>7d}/{r['n_probes']}")
ctl = [r for r in results if r["arm"] == "control_frozen"][0]
print(f"\nCONTROL VALID: {ctl['drift'] == 0.0 and ctl['updates'] == 0}")
print("wrote results.json")

## 10 · Talk to it while it runs

The point of the whole thing. The session is mid-loop; a request is answered
against the weights **as they are right now**, including whatever was committed
seconds ago.

In [ ]:
model, tok, mems = build(cfg)
model.eval()
collect_background(model, tok, mems, BACKGROUND, repeats=10)
w = Writer(model, tok, mems, cfg)

topic = "Product key memory"
text = wiki(topic)
print("BEFORE:", generate(model, tok, f"In one sentence, what is {topic}?", 40))
if len(text) > 120:
    before = w.nll(text)
    kept = w.learn(text, [])
    print(f"\nlearned '{topic}': NLL {before:.4f} -> {w.nll(text):.4f}  kept={kept}")
print("\nAFTER: ", generate(model, tok, f"In one sentence, what is {topic}?", 40))

## 11 · Checkpoint and resume

Colab disconnects. What you get is checkpoint-and-resume, not immortality —
saying otherwise would be dishonest about what "never stops" can mean on a
free tier.

In [ ]:
def save(mems, path="session.pt"):
    torch.save({li: {"V": m.V.weight.detach().cpu(),
                     "alpha": m.alpha.detach().cpu(),
                     "read": m.read_counts.cpu(),
                     "bg": m.bg_counts.cpu()} for li, m in mems.items()}, path)
    print(f"saved {path} ({os.path.getsize(path)/1e6:.1f} MB)")

def load(mems, path="session.pt"):
    sd = torch.load(path, map_location=DEV)
    with torch.no_grad():
        for li, m in mems.items():
            m.V.weight.copy_(sd[li]["V"].to(DEV))
            m.alpha.copy_(sd[li]["alpha"].to(DEV))
            m.read_counts.copy_(sd[li]["read"].to(DEV))
            m.bg_counts.copy_(sd[li]["bg"].to(DEV))
    print("resumed")

save(mems)

## 12 · What this is and is not

**Is:** a 0.5B model on one T4 that runs continuously, reads when it finds a
gap, writes what it reads into weights, verifies each write before keeping it,
and forgets substantially less than LoRA or full finetuning would at the same
level of acquisition. Every write is auditable and every claim above is
measured.

**Is not:**

- *Not* "all weights active." The backbone stays frozen. ~90% of parameters are
  still not writable. Calling this an unfrozen brain would be false.
- *Not* forgetting eliminated. Reduced, and by a lot, but it compounds over
  hours.
- *Not* more intelligent. More knowledgeable. Reasoning stays 0.5B-level.
- *Not* immortal. Colab disconnects; you resume from a checkpoint.

The honest headline: **it remembers what it reads, and forgets far less than
anything you can get from LoRA or full finetuning at the same acquisition.**
That is a real, defensible result. It is not AGI, and the gap between the two
is not a matter of scaling this up.